# 06 · Verificación en notebook limpio
Carga el modelo desde `models:/resenas-bancos-ec-modelo@champion` y comprueba que reproduce **exactamente** la métrica de prueba registrada en MLflow. Ejecutar en un kernel nuevo (*Restart* → *Run All*), después del `05`.

In [ ]:
import mlflow, mlflow.sklearn, numpy as np
from sklearn.metrics import average_precision_score
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import NOMBRE_MODELO, configurar
configurar()
cli = mlflow.MlflowClient()
mv = cli.get_model_version_by_alias(NOMBRE_MODELO, "champion")
run = cli.get_run(mv.run_id)
modelo = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}@champion")
print("Modelo:", NOMBRE_MODELO, "| versión", mv.version, "| run_id", mv.run_id)

In [ ]:
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)
p = modelo.predict_proba(test[FEATURES])[:, 1]
umbral = run.data.metrics["umbral_aplicado"]
pr_auc = average_precision_score(test[TARGET], p)
costo = costo_negocio(test[TARGET], (p >= umbral).astype(int))
print(f"PR-AUC prueba  → reproducida: {pr_auc:.6f} | registrada: {run.data.metrics['test_pr_auc']:.6f}")
print(f"Costo negocio  → reproducido: {costo:.6f} | registrado: {run.data.metrics['test_costo']:.6f}")
assert abs(pr_auc - run.data.metrics["test_pr_auc"]) < 1e-9 and abs(costo - run.data.metrics["test_costo"]) < 1e-9
print("OK: el modelo reproduce exactamente las métricas registradas.")